# Lesson 4 — Ingesting the Nigeria MTF Survey (first lesson in VS Code)

**Goal today:** extract the MTF survey data, load our first files, and make one important discovery about our target variable.

**How to work:** read each markdown cell, then run the code cell below it with **Shift+Enter**. Cells marked **YOUR TURN** are yours to complete. If anything confuses you, stop and ask your mentor — that is the system working, not failing.

## Step 1 — Imports

Remember: Python starts empty-handed. We fetch three toolboxes:
- `zipfile` — opens ZIP archives (built into Python, nothing to install)
- `pathlib.Path` — the modern way to handle file paths (also built in)
- `pandas` — our data workhorse

In [ ]:
import zipfile
from pathlib import Path
import pandas as pd

print("Toolboxes loaded.")

## Step 2 — Extract the raw data

We unzip *with code* rather than by right-clicking, so the notebook documents every step — anyone (including future you) can reproduce it from scratch.

Note: this notebook lives in `notebooks/`, so we reach the data with `../raw_data/` — meaning "go up one folder, then into raw_data".

In [ ]:
RAW = Path("../raw_data")
EXTRACTED = RAW / "extracted"
EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(RAW / "raw-data_nigeria.zip") as z:
    z.extractall(EXTRACTED)

print("Extraction complete.")

## Step 3 — See what we received

A `.dta` file is Stata's data format — very common for survey microdata from the World Bank and DHS. pandas reads it directly with `pd.read_stata()`, and it even preserves the *labels* (the human-readable answer texts), which you'll appreciate as a survey researcher.

In [ ]:
dta_files = sorted(EXTRACTED.rglob("*.dta"))
for f in dta_files:
    size_kb = f.stat().st_size // 1024
    print(f"{f.name:45s} {size_kb:>8,} KB")

Each file is one **section of the questionnaire** (open `docs/mtf_nigeria_hh_questionnaire.pdf` and you'll see the same letters: Section A = identification, B = roster, C = electricity, F = fuels...). The survey was split exactly the way you'd split a paper questionnaire.

## Step 4 — Load our first file: household identification

This file should contain one row per household with location information — including, if your instinct is right, the urban/rural variable you bet on.

In [ ]:
ident = pd.read_stata(next(f for f in dta_files if "Identification" in f.name))
print("Shape (rows, columns):", ident.shape)
ident.head()

## Step 5 — YOUR TURN: explore

Use the tools you know from your churn project. In the empty cells below, find out:
1. The column names — which column looks like urban/rural? (`ident.columns`)
2. How many households are urban vs rural? (`.value_counts()` on that column)
3. Are there duplicate households? (`ident["HH_ID"].duplicated().sum()`)

In [ ]:
# YOUR TURN — column names


In [ ]:
# YOUR TURN — urban vs rural counts


In [ ]:
# YOUR TURN — duplicate check


## Step 6 — The solar file, and a discovery

Our project is about solar adoption, and there is a file dedicated to solar devices. Load it and look carefully at its **size**.

In [ ]:
solar = pd.read_stata(next(f for f in dta_files if "SOLAR" in f.name))
print("Shape:", solar.shape)
print("Unique households owning a solar device:", solar["HH_ID"].nunique())
solar.head()

## Step 7 — Think before next lesson

Compare two numbers: households in the identification file (Step 4) versus unique households in the solar file (Step 6).

**Question for your mentor session:** if we define our target as "household owns a solar device," what proportion of households are adopters? Why might that be a problem for a model — and does it remind you of anything from your financial inclusion project?

There is no wrong answer here. Bring your thinking, not a polished result.

---

**Before closing:** Kernel → Restart Kernel and Run All Cells. If everything runs clean, commit:
```
git add notebooks/01_data_ingestion.ipynb
git commit -m "Lesson 4: extract MTF data, first exploration"
```